# Titanic exp011 - Add TicketGroupSize on top of Title + FamilyGroup + SexPclass

この notebook は現在のベスト構成 `Title + FamilyGroup + SexPclass` に `TicketGroupSize` を追加して比較するためのものです。

仮説:
- 同じ `Ticket` を共有する人数は、同行グループの規模や予約単位の違いを表しているかもしれない
- `Title`、`FamilyGroup`、`SexPclass` では取り切れない集団行動の差を `TicketGroupSize` で補えるかもしれない


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    COMP_DIR = cwd.parent
elif cwd.name == "titanic":
    COMP_DIR = cwd
else:
    COMP_DIR = Path("/home/sora/dev/kaggle/competitions/titanic")

DATA_DIR = COMP_DIR / "data"
SUBMISSION_DIR = COMP_DIR / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
OUTPUT_PATH = SUBMISSION_DIR / "exp011_ticketgroupsize.csv"

FEATURE_COLUMNS = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "Title", "FamilyGroup", "SexPclass", "TicketGroupSize"]
NUMERIC_COLUMNS = ["Age", "SibSp", "Parch", "Fare", "TicketGroupSize"]
CATEGORICAL_COLUMNS = ["Pclass", "Sex", "Embarked", "Title", "FamilyGroup", "SexPclass"]

TITLE_NORMALIZATION = {
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs",
    "Lady": "Rare",
    "Countess": "Rare",
    "Dona": "Rare",
    "Dr": "Rare",
    "Rev": "Rare",
    "Col": "Rare",
    "Major": "Rare",
    "Capt": "Rare",
    "Sir": "Rare",
    "Don": "Rare",
    "Jonkheer": "Rare",
    "Master": "Master",
    "Miss": "Miss",
    "Mr": "Mr",
    "Mrs": "Mrs",
}


In [ ]:
def add_features(train_df: pd.DataFrame, test_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    combined = pd.concat(
        [
            train_df.copy().assign(_dataset="train"),
            test_df.copy().assign(_dataset="test"),
        ],
        ignore_index=True,
        sort=False,
    )

    title = combined["Name"].fillna("").astype(str).str.extract(r",\s*([^\.]+)\.", expand=False)
    combined["Title"] = title.fillna("Rare").map(lambda x: TITLE_NORMALIZATION.get(x, "Rare")).astype(str)
    combined["FamilySize"] = combined["SibSp"].fillna(0) + combined["Parch"].fillna(0) + 1
    combined["FamilyGroup"] = pd.cut(
        combined["FamilySize"],
        bins=[0, 1, 4, 100],
        labels=["alone", "small", "large"],
        right=True,
    ).astype(str)
    combined["SexPclass"] = combined["Sex"].fillna("missing").astype(str) + "_" + combined["Pclass"].fillna(-1).astype(int).astype(str)

    ticket_key = combined["Ticket"].fillna("missing").astype(str)
    ticket_counts = ticket_key.value_counts()
    combined["TicketGroupSize"] = ticket_key.map(ticket_counts).astype(float)

    train_features = combined.loc[combined["_dataset"] == "train"].drop(columns=["_dataset"])
    test_features = combined.loc[combined["_dataset"] == "test"].drop(columns=["_dataset"])
    return train_features, test_features


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
train_df, test_df = add_features(train_df, test_df)

display(train_df[["Ticket", "TicketGroupSize", "SexPclass", "Title", "FamilyGroup"]].head())
display(train_df.groupby("TicketGroupSize", observed=False)["Survived"].agg(["mean", "count"]).sort_index())


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            NUMERIC_COLUMNS,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", make_one_hot_encoder()),
                ]
            ),
            CATEGORICAL_COLUMNS,
        ),
    ],
    remainder="drop",
    sparse_threshold=0.0,
)

pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=3000,
                C=1.0,
                solver="lbfgs",
                random_state=42,
            ),
        ),
    ]
)


In [ ]:
X = train_df[FEATURE_COLUMNS].copy()
y = train_df["Survived"].astype(int)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy", n_jobs=-1)

print("CV scores:", [round(score, 5) for score in scores])
print("CV mean:", round(scores.mean(), 5))
print("CV std:", round(scores.std(), 5))


比較メモ:

- `exp001` baseline CV mean: `0.79687`
- `exp006` Title + FamilyGroup CV mean: `0.82939`
- `exp007` Title + FamilyGroup + SexPclass CV mean: `0.83614`
- `exp011` exp007 + TicketGroupSize CV mean: 実行結果を記入


In [ ]:
pipeline.fit(X, y)
test_predictions = pipeline.predict(test_df[FEATURE_COLUMNS].copy()).astype(int)

submission_df = pd.DataFrame(
    {
        "PassengerId": test_df["PassengerId"],
        "Survived": test_predictions,
    }
)
submission_df.to_csv(OUTPUT_PATH, index=False)

print("saved:", OUTPUT_PATH)
display(submission_df.head())
